# Exploratory Data Analysis (EDA) & Encoding - E3
Mainly, this notebook covers EDA on the raw trace data and encoding it for use in other steps of the development process.
Also this notebook will compare the performance of using the both method, Which is more robust to go forward without losing any data in the process

In [1]:
import sys
import os
import time
import json
import pandas as pd
import numpy as np
from collections import Counter

sys.path.append("..")

# Import the function from the utils which will needed here
from utils import (
    stream_opcodes,
    count_opcodes_from_stream,
    build_opcode_vocab,
    encode_trace_to_npy,
    save_vocab,
    load_vocab,
    check_vocab_size,
    decode_trace_npy,
    load_encoded_trace
)

## Initializing the Paths

In [2]:
RAW_TRACES_DIR = "../sandbox/data-e3/raw_traces"
MANIFEST_CSV = "../sandbox/data-e3/manifest.csv"

# Load the manifest CSV to get the metadata to use later
df_manifest = pd.read_csv(MANIFEST_CSV)
print(f"Manifest Loaded: {len(df_manifest)} rows")

# Get the list of raw trace files in the directory and sort them for processing
trace_files = sorted([
    os.path.join(RAW_TRACES_DIR, f)
    for f in os.listdir(RAW_TRACES_DIR)
    if f.endswith(".txt")
])
print(f"Raw trace files found: {len(trace_files)}")

Manifest Loaded: 100 rows
Raw trace files found: 100


----
## EDA using Raw Trace Streaming - Method 1
Each of the raw opcode trace file is streamed line by line using `stream_opcodes()` function. A single pass over each file simultaneously to improve the performance.

In [3]:
# Set the data structures to hold the global vocab, per file opcode counts, and streaming times
global_vocab = set()
per_file_counts = {}
stream_times = {}

MIN_OPCODE_COUNT = 10 # Filter threshold to removes likely malformed or truncated opcodes

print(f"Streaming {len(trace_files)} trace files...")
print("This will take a moment base on the given file sizes. The progress shown per 10 files\n")

total_start = time.time()

# Stream through each trace file count the opcodes and store the counts and elapsed time
for i, path in enumerate(trace_files, 1):
    filename = os.path.basename(path)
    t0 = time.time()

    counter = Counter()
    for opcode in stream_opcodes(path):
        counter[opcode] += 1

    elapsed = time.time() - t0
    stream_times[filename] = elapsed
    per_file_counts[filename] = counter

    if i % 10 == 0 or i == len(trace_files):
        done = time.time() - total_start
        print(f"  [{i:>3}/{len(trace_files)}]  elapsed: {done/60:.1f} min")

# Build global counts then filter by minimum threshold
global_counter = Counter()
for c in per_file_counts.values():
    global_counter.update(c)

# Filter the global counter to remove opcodes that are below the minimum threshold
removed = [op for op, count in global_counter.items() if count < MIN_OPCODE_COUNT]
global_vocab = {op for op, count in global_counter.items() if count >= MIN_OPCODE_COUNT}

total_stream_time = time.time() - total_start
print(f"\nStreaming Complete:  {total_stream_time/60:.1f} min total")
print(f"Raw Vocabulary Size: {len(global_counter)} distinct opcodes")
print(f"After Filtering (>={MIN_OPCODE_COUNT}): {len(global_vocab)} distinct opcodes")
print(f"Removed {len(removed)}")

Streaming 100 trace files...
This will take a moment base on the given file sizes. The progress shown per 10 files

  [ 10/100]  elapsed: 15.5 min
  [ 20/100]  elapsed: 30.0 min
  [ 30/100]  elapsed: 44.9 min
  [ 40/100]  elapsed: 62.3 min
  [ 50/100]  elapsed: 76.6 min
  [ 60/100]  elapsed: 90.0 min
  [ 70/100]  elapsed: 105.7 min
  [ 80/100]  elapsed: 121.3 min
  [ 90/100]  elapsed: 135.4 min
  [100/100]  elapsed: 149.8 min

Streaming Complete:  149.9 min total
Raw Vocabulary Size: 756845 distinct opcodes
After Filtering (>=10): 83329 distinct opcodes
Removed 673516


### Opcode Vocabulary
The full set of distinct x86 opcode mnemonics seen across all 260 files. `check_vocab_size` confirms the count is safely within the `int32` range used for encoding.

In [4]:
vocab = sorted(global_vocab)
check_vocab_size(vocab)

print(f"Vocabulary Size: {len(vocab)} distinct opcodes")
# print(f"Opcodes List: {vocab}")

Vocabulary Size: 83329 distinct opcodes


### Crosscheck opcode counts with the manifest data

The manifest already has `opcode_count` from the log's "Opcodes recorded" line. Comparing these with the actual line counts from the streaming pass confirms the raw trace files are complete and uncorrupted.

In [5]:
stream_counts = {
    fname: sum(c.values())
    for fname, c in per_file_counts.items()
}

df_cross = df_manifest[["filename", "opcode_count"]].copy()
df_cross["stream_count"] = df_cross["filename"].map(stream_counts)
df_cross["match"] = df_cross["opcode_count"] == df_cross["stream_count"]

mismatches = df_cross[~df_cross["match"].fillna(False)]
print(f"Files with matching counts: {df_cross['match'].sum()}")
print(f"Mismatches or missing count: {len(mismatches)}")

if len(mismatches):
    print("\nMismatched Files:")
    print(mismatches)

Files with matching counts: 0
Mismatches or missing count: 100

Mismatched Files:
                  filename  opcode_count  stream_count  match
0         abbot-0.13.0.jar     249594646           NaN  False
1      activemq-4.0-M1.jar     248111879           NaN  False
2   activemq-run-6.2.6.jar     260239525           NaN  False
3    ant-contrib-1.0b2.jar     140904968           NaN  False
4   ant-launcher-1.6.5.jar     170001408           NaN  False
..                     ...           ...           ...    ...
95   tomcat-jni-5.0.28.jar     148010387           NaN  False
96    webservice-0.5.1.jar     254863458           NaN  False
97       webwork-2.2.5.jar     197700531           NaN  False
98           which-1.0.jar     248361141           NaN  False
99    xml-resolver-1.2.jar     185150149           NaN  False

[100 rows x 4 columns]


### Global Opcode Frequency Distribution

Analyze the frequency across all 260 files, which opcodes dominate the instruction mix. This clearly shows later feature engineering decisions.

This is a summary table only for this stage, plots will be created on later step.

In [6]:
global_counter = Counter()

for count in per_file_counts.values():
    global_counter.update(count)

df_freq = pd.DataFrame(
    global_counter.most_common(),
    columns=["opcode", "total_count"]
)

df_freq["proportion"] = df_freq["total_count"] / df_freq["total_count"].sum()

print(f"Top 20 most frequent opcodes across all {len(trace_files)} files:\n")
print(df_freq.head(20).to_string(index=False))

Top 20 most frequent opcodes across all 100 files:

   opcode  total_count  proportion
      MOV   6798364056    0.311864
      CMP   1726031187    0.079179
       JZ   1208031252    0.055416
      ADD   1195134730    0.054825
     PUSH   1092971569    0.050138
     TEST   1036236783    0.047536
      POP    821584903    0.037689
      LEA    759476905    0.034840
      JNZ    735659042    0.033747
    MOVZX    632609055    0.029020
      JMP    576978644    0.026468
      NOP    425153100    0.019503
      AND    410920719    0.018850
      SUB    399797157    0.018340
 RET_NEAR    358048337    0.016425
CALL_NEAR    352399040    0.016166
   MOVSXD    261726171    0.012006
    STOSB    256149359    0.011750
      SHR    236369714    0.010843
      XOR    228724943    0.010492


#### Bottom of the distribution to list the rarest opcodes

In [7]:
print("Rarest 20 opcodes (lowest global frequency):\n")
print(df_freq.tail(20).to_string(index=False))

Rarest 20 opcodes (lowest global frequency):

             opcode  total_count   proportion
         JLEJMPCMOV            1 4.587336e-11
              ROJNL            1 4.587336e-11
           MMOVUPSA            1 4.587336e-11
             CMPDEA            1 4.587336e-11
       JNSCMOL_NEAR            1 4.587336e-11
        JZTESTZPUSH            1 4.587336e-11
          MOVANDSAR            1 4.587336e-11
          JNSMOVZXP            1 4.587336e-11
      SUBJZAMOVAPSM            1 4.587336e-11
   MOVAPSPUNPCKLDQR            1 4.587336e-11
     LEASHANDMOVSXD            1 4.587336e-11
    PUSHPUSHMMOVSXD            1 4.587336e-11
        CALL_RDTSCP            1 4.587336e-11
MOVAPSCMPXCHG_LOCKX            1 4.587336e-11
             MOLREA            1 4.587336e-11
          MOVUNMLEA            1 4.587336e-11
     PUNPCKLDQLTEST            1 4.587336e-11
        RET_TESTEAR            1 4.587336e-11
             JLETEA            1 4.587336e-11
    MOVZXCMPSMOVSXD            1 4

### Check per file trace length distribution

How much do the 260 files vary in total opcode counts, large variation matters for normalization and some file have 260M opcodes and some are 50M opcodes, this should contribute comparable feature vectors, not be dominated by raw size.

In [8]:
df_lengths = pd.DataFrame({
    "filename": list(stream_counts.keys()),
    "opcode_count": list(stream_counts.values()),
})

print(df_lengths["opcode_count"].describe().apply(lambda x: f"{x:,.0f}"))

count            100
mean     217,991,443
std       46,831,168
min      132,335,910
25%      173,503,166
50%      248,124,172
75%      257,648,664
max      267,970,136
Name: opcode_count, dtype: str


### Streaming Timing Summary

Time taken per file using raw text streaming, the baseline for the performance comparison with Method 2 (Encoded Array) later.

In [9]:
df_timing = pd.DataFrame({
    "filename": list(stream_times.keys()),
    "stream_sec": list(stream_times.values()),
})

df_timing["opcodes"] = df_timing["filename"].map(stream_counts)
df_timing["lines_per_sec"] = (df_timing["opcodes"] / df_timing["stream_sec"]).astype(int)

print(f"Streaming Speed Summary (Method 1)")
print(df_timing["lines_per_sec"].describe().apply(lambda x: f"{x:,.0f}"))
print(f"\nTotal Streaming Time: {df_timing['stream_sec'].sum()/60:.1f} min")

Streaming Speed Summary (Method 1)
count          100
mean     2,426,240
std         65,940
min      2,154,921
25%      2,397,608
50%      2,431,398
75%      2,459,433
max      2,570,724
Name: lines_per_sec, dtype: str

Total Streaming Time: 149.8 min


### Save EDA Outputs into CSVs

Save the frequency table and timing results as CSVs so it can be used for later visualization and later notebooks can load them without re-streaming.

In [10]:
os.makedirs("../sandbox/data-e3/processed_outputs", exist_ok=True)

df_freq.to_csv("../sandbox/data-e3/processed_outputs/global_opcode_freq.csv", index=False)
print("global_opcode_freq.csv file saved!")

df_lengths.to_csv("../sandbox/data-e3/processed_outputs/per_file_lengths.csv", index=False)
print("per_file_lengths.csv file saved!")

df_timing.to_csv("../sandbox/data-e3/processed_outputs/streaming_timing.csv", index=False)
print("streaming_timing.csv file saved!")

global_opcode_freq.csv file saved!
per_file_lengths.csv file saved!
streaming_timing.csv file saved!


----
### FIX: Crosscheck opcode counts with the manifest data
The cross-check mismatch is actually a filename extension bug, the manifest.csv has `.jar` filenames but stream_counts keys are `.txt` filenames, so .map() finds nothing. Not a data problem.

In [11]:
# Fix: strip .txt and add .jar to match manifest filename format
stream_counts_jar = {
    fname.replace(".txt", ".jar"): count
    for fname, count in stream_counts.items()
}

df_cross = df_manifest[["filename", "opcode_count"]].copy()
df_cross["stream_count"] = df_cross["filename"].map(stream_counts_jar)
df_cross["match"] = df_cross["opcode_count"] == df_cross["stream_count"]

mismatches = df_cross[~df_cross["match"].fillna(False)]
print(f"Files with matching counts: {df_cross['match'].sum()}")
print(f"Mismatches or missing: {len(mismatches)}")

if len(mismatches):
    print(f"\nMismatched File:\n{mismatches}")

Files with matching counts: 0
Mismatches or missing: 100

Mismatched File:
                  filename  opcode_count  stream_count  match
0         abbot-0.13.0.jar     249594646     249459514  False
1      activemq-4.0-M1.jar     248111879     247994762  False
2   activemq-run-6.2.6.jar     260239525     260119275  False
3    ant-contrib-1.0b2.jar     140904968     140835363  False
4   ant-launcher-1.6.5.jar     170001408     169894795  False
..                     ...           ...           ...    ...
95   tomcat-jni-5.0.28.jar     148010387     147946742  False
96    webservice-0.5.1.jar     254863458     254749839  False
97       webwork-2.2.5.jar     197700531     197604164  False
98           which-1.0.jar     248361141     248253581  False
99    xml-resolver-1.2.jar     185150149     185065771  False

[100 rows x 4 columns]


The fix actually worked - stream_count is now populated with real numbers, not NaN anymore. There is a tiny difference that is expected and not a problem. 
Because the manifest opcode_count comes from run_trace.sh which uses `wc -l` to count lines. So `wc -l` counts every new line character including empty lines. But the stream_opcodes function skips empty lines. So any empty lines in the .txt file get counted by `wc -l` but skipped by the streamer.

### Fix the crosscheck to use approximate matching instead of exact equal

In [12]:
# Allow a small tolerance value to account for empty lines
TOLERANCE = 0.001

df_cross["diff_pct"] = abs(df_cross["opcode_count"] - df_cross["stream_count"]) / df_cross["opcode_count"]
df_cross["match"] = df_cross["diff_pct"] <= TOLERANCE

mismatches = df_cross[~df_cross["match"].fillna(False)]
print(f"Files within {TOLERANCE*100}% Tolerance: {df_cross['match'].sum()}")
print(f"Files Outside Tolerance: {len(mismatches)}")
print(f"\nMax Difference across all files: {df_cross['diff_pct'].max()*100:.4f}%")
print(f"Mean Difference across all files: {df_cross['diff_pct'].mean()*100:.4f}%")

Files within 0.1% Tolerance: 100
Files Outside Tolerance: 0

Max Difference across all files: 0.0627%
Mean Difference across all files: 0.0445%


### E3 Vocabulary coverage check against the fixed E1 287-opcode vocab

In [13]:
# The vocab is not rebuilt for E2 - it stays fixed
# so feature vectors remain directly comparable across environments
E1_CLEAN_VOCAB = "../sandbox/data/vocab_clean.json"
clean_vocab = load_vocab(E1_CLEAN_VOCAB)
print(f"E1 clean vocab loaded: {len(clean_vocab)} opcodes")

covered = df_freq[df_freq.opcode.isin(clean_vocab)]
total_e2 = df_freq.total_count.sum()
kept_e2 = covered.total_count.sum()

print(f"\nE2 stream covered by fixed 287 vocab: {kept_e2/total_e2*100:.4f}%")
print(f"E2 stream not covered: {(total_e2-kept_e2)/total_e2*100:.4f}%")
print(f"Vocab Entries present in E2: {len(covered)} of {len(clean_vocab)}")

# Opcodes in the fixed vocab that never appear in E2
absent = sorted(set(clean_vocab) - set(covered.opcode))
print(f"\nVocab opcodes absent from E2 ({len(absent)}): {absent}")

# Highest-frequency E2 opcodes NOT in the fixed vocab
missing = df_freq[~df_freq.opcode.isin(clean_vocab)].head(30)
print(f"\nTop 30 E2 opcodes outside the fixed vocab:")
print(missing.to_string(index=False))

E1 clean vocab loaded: 287 opcodes

E2 stream covered by fixed 287 vocab: 99.7830%
E2 stream not covered: 0.2170%
Vocab Entries present in E2: 274 of 287

Vocab opcodes absent from E2 (13): ['CMPSW', 'LFENCE', 'LOOP', 'MOVNTDQ', 'MOVNTPS', 'PSUBD', 'SETNO', 'SFENCE', 'SHUFPS', 'VMOVHPD', 'VMOVLPD', 'VMOVNTDQ', 'VPUNPCKHQDQ']

Top 30 E2 opcodes outside the fixed vocab:
 opcode  total_count  proportion
     OV      1855452    0.000085
 MOVMOV      1621837    0.000074
      V       805679    0.000037
     MP       657147    0.000030
     MO       549365    0.000025
      P       519565    0.000024
      M       429068    0.000020
      Z       401824    0.000018
 CMPMOV       390124    0.000018
 MOVCMP       389097    0.000018
     OP       378044    0.000017
    USH       297362    0.000014
    EST       275227    0.000013
     DD       269680    0.000012
PUSHMOV       268539    0.000012
  JZMOV       262942    0.000012
MOVPUSH       251671    0.000012
   MMOV       246153    0.000011
  